# Training LoRA Qwen untuk KKA — Cloud GPU (Jupyter)

Notebook ini untuk instance GPU sewaan dengan template **PyTorch + Jupyter** (Ubuntu),
misalnya RTX 5090 / 4090 / 3090 / A100. Untuk Google Colab tetap pakai
`Training_LoRA_Colab.ipynb`.

Perbedaan dengan versi Colab:
- Tidak memakai Google Drive / `google.colab`; semua file ada di disk instance.
- Ada pemeriksaan arsitektur GPU (RTX 50xx/Blackwell butuh PyTorch build CUDA ≥ 12.8).
- Mendukung `--precision bf16`, `--batch-size`, dan `--grad-accum` pada `training/train_lora.py`.

> Disk instance **hilang saat instance dihapus**. Unduh adaptor final (langkah 7) sebelum masa sewa habis.

Jalankan sel secara berurutan.

## 1. Masukkan proyek ke instance

Pilih salah satu, lewat **Terminal** Jupyter:

**A. Git (disarankan)**
```bash
git clone https://github.com/<username>/kka_lab.git
```

**B. ZIP** — di Windows, kompres folder `kka_lab` **tanpa** `.venv` (venv Windows tidak jalan di Linux),
lalu drag-and-drop `kka_lab.zip` ke panel file Jupyter. Sel berikut mengekstraknya otomatis.

Jangan unggah `.env` berisi token Astra; training tidak membutuhkannya.

In [ ]:
import os
import zipfile
from pathlib import Path

# Sesuaikan jika proyek diletakkan di folder lain.
PROJECT = Path('kka_lab').resolve()

zip_path = Path('kka_lab.zip')
if not PROJECT.exists() and zip_path.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('.')
    print('ZIP diekstrak.')

assert (PROJECT / 'training/train_lora.py').exists(), f'Proyek tidak ditemukan di {PROJECT}'
assert not (PROJECT / '.venv/Scripts').exists(), 'Hapus folder .venv Windows dari proyek ini.'
os.chdir(PROJECT)
print('Folder kerja:', Path.cwd())

## 2. Periksa GPU dan PyTorch

**Jangan menginstal ulang torch** dari README (`cu126`) — build itu untuk laptop RTX 3050 dan
tidak memiliki kernel untuk RTX 50xx (sm_120). Pakai torch bawaan template.
Sel ini berhenti jika build torch tidak mendukung GPU yang terpasang.

In [ ]:
import subprocess
import torch

assert torch.cuda.is_available(), 'GPU belum terdeteksi. Pilih template PyTorch dengan GPU.'
major, minor = torch.cuda.get_device_capability(0)
arch = f'sm_{major}{minor}'
arch_list = torch.cuda.get_arch_list()
props = torch.cuda.get_device_properties(0)

print('GPU          :', torch.cuda.get_device_name(0))
print('VRAM GiB     :', round(props.total_memory / 2**30, 1))
print('Compute cap. :', arch)
print('PyTorch      :', torch.__version__, '| CUDA build:', torch.version.cuda)
print('Arsitektur   :', arch_list)
print('bf16 support :', torch.cuda.is_bf16_supported())

if major >= 12:  # Blackwell konsumen (RTX 50xx)
    cuda_build = tuple(int(x) for x in (torch.version.cuda or '0.0').split('.')[:2])
    assert cuda_build >= (12, 8), (
        f'{arch} butuh PyTorch build CUDA >= 12.8, terpasang {torch.version.cuda}. '
        'Ganti template ke PyTorch terbaru (cu128 atau lebih baru).'
    )
assert arch in arch_list or f'compute_{major}{minor}' in arch_list, (
    f'Build torch ini tidak dikompilasi untuk {arch}. Ganti template PyTorch yang lebih baru.'
)

# Uji kernel GPU sungguhan (gagal di sini = build torch tidak cocok).
x = torch.randn(256, 256, device='cuda', dtype=torch.bfloat16)
torch.cuda.synchronize()
print('Uji matmul   :', float((x @ x).float().abs().mean()))
TORCH_BEFORE = torch.__version__
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

## 3. Instal dependensi proyek

`requirements-core.txt` tidak memuat torch, sehingga torch bawaan template dipertahankan.
Sel ini memeriksa ulang bahwa versi torch tidak berubah.

In [ ]:
%pip install -q -r requirements-core.txt

In [ ]:
import importlib.metadata as md_

torch_after = md_.version('torch')
print('torch sebelum:', TORCH_BEFORE, '| sesudah:', torch_after)
assert torch_after.split('+')[0] == TORCH_BEFORE.split('+')[0], (
    'pip mengganti versi torch. Instal ulang torch bawaan template, lalu restart kernel.'
)
for pkg in ['transformers', 'peft', 'accelerate', 'bitsandbytes']:
    print(f'{pkg:13}:', md_.version(pkg))
print('\nJika ada paket yang baru di-upgrade, pilih Kernel > Restart Kernel, '
      'lalu jalankan sel 1 dan 2 lagi sebelum lanjut.')

## 4. Uji bitsandbytes 4-bit di GPU

QLoRA memakai kuantisasi NF4 dari bitsandbytes. Paket PyPI untuk CUDA 12.8 ke atas
memuat kernel sm_100/sm_120 (Blackwell). Uji kecil ini memastikan kernel 4-bit benar-benar
jalan sebelum mengunduh model ~6 GB.

In [ ]:
import torch
import bitsandbytes as bnb

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
layer = bnb.nn.Linear4bit(
    128, 128, bias=False, compute_dtype=dtype, quant_type='nf4'
).to('cuda')
out = layer(torch.randn(4, 128, device='cuda', dtype=dtype))
torch.cuda.synchronize()
print('bitsandbytes', bnb.__version__, 'OK | output', tuple(out.shape), out.dtype)

## 5. Siapkan dataset dan record model

- `model_record.json` harus hasil `scripts/download_model.py` (ID model + revisi persis).
- Dataset penelitian: `data/lora_dialogues.jsonl` (unggah lewat panel file Jupyter).
- `data/demo_dialogues.jsonl` hanya untuk uji alur; hasilnya **bukan** adaptor penelitian.

Sel ini memvalidasi dataset **sebelum** model diunduh, jadi kesalahan data ketahuan lebih awal.

In [ ]:
import json
import sys
from pathlib import Path

DATA = Path('data/lora_dialogues.jsonl')   # ganti ke data/demo_dialogues.jsonl untuk uji alur
RECORD = Path('model_record.json')

assert DATA.exists(), f'Dataset {DATA} belum ada. Unggah ke folder data/.'
assert RECORD.exists(), 'model_record.json belum ada.'
ALLOW_DEMO = 'demo' in DATA.name.lower()

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
from app.features.dataset import split_groups, validate_rows

rows = [json.loads(x) for x in DATA.read_text(encoding='utf-8').splitlines() if x.strip()]
validate_rows(rows)
train_rows, val_rows = split_groups(rows)
print('Record model :', json.loads(RECORD.read_text(encoding='utf-8')))
print('Total baris  :', len(rows), '| train:', len(train_rows), '| validation:', len(val_rows))
print('Group unik   :', len({r['group_id'] for r in rows}))
if ALLOW_DEMO:
    print('\nPERINGATAN: data demo -> --allow-demo dipakai. Adaptor hanya untuk uji alur.')

## 6. Latih adaptor

Pengaturan untuk GPU besar (≥ 24 GB): `bf16`, batch 4, gradient accumulation 2
(effective batch = 8, sama dengan konfigurasi Colab). Jika muncul CUDA out of memory,
turunkan `BATCH_SIZE` ke 2 dan naikkan `GRAD_ACCUM` ke 4.

> Catatan penelitian: presisi (`bf16`/`fp16`) dan batch tercatat di `run_manifest.json`.
> Jika membandingkan beberapa run, samakan pengaturan ini.

Output di-stream ke sel. Untuk run panjang, lebih aman menjalankan perintah yang dicetak sel ini
di Terminal dalam `tmux` agar training tetap berjalan walau browser terputus.

In [ ]:
import shlex
import subprocess
from datetime import datetime

import torch

PRECISION = 'bf16' if torch.cuda.is_bf16_supported() else 'fp16'
BATCH_SIZE = 4
GRAD_ACCUM = 2
RUN_NAME = datetime.now().strftime('run_%Y%m%d_%H%M')
OUT = Path('adapters') / RUN_NAME
RESUME = ''   # isi path checkpoint untuk melanjutkan, lihat sel markdown berikut

cmd = [
    sys.executable, 'training/train_lora.py',
    '--data', str(DATA), '--record', str(RECORD), '--out', str(OUT),
    '--precision', PRECISION,
    '--batch-size', str(BATCH_SIZE), '--grad-accum', str(GRAD_ACCUM),
]
if ALLOW_DEMO:
    cmd.append('--allow-demo')
if RESUME:
    cmd += ['--resume', RESUME]

print('Perintah setara untuk Terminal/tmux:\n')
print('cd', shlex.quote(str(Path.cwd())), '&&', shlex.join(['python', *cmd[1:]]), '\n')

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
assert proc.returncode == 0, f'Training gagal (exit code {proc.returncode}). Lihat log di atas.'
print('\nSelesai:', OUT)

### Jika training terputus

Isi `RESUME` dengan checkpoint yang benar-benar ada, misalnya
`adapters/run_YYYYMMDD_HHMM/checkpoints/checkpoint-N`, set `RUN_NAME` ke nama run yang sama,
lalu jalankan ulang sel di atas. Jangan resume jika data, model, atau konfigurasi telah diubah.

In [ ]:
for ckpt in sorted(Path('adapters').glob('*/checkpoints/checkpoint-*')):
    print(ckpt)

## 7. Periksa hasil dan kemas untuk diunduh

Sel ini membuat dua ZIP di folder `exports/`:
- `<run>-qwen-kka-lora.zip` — adaptor final (puluhan MB). Isinya folder `qwen-kka-lora/`.
- `<run>-records.zip` — catatan penelitian (`run_manifest`, `packages.lock`, `evaluation`,
  `trainer_state`). Checkpoint tidak ikut karena besar.

Unduh keduanya: klik kanan file di panel Jupyter → **Download**.

In [ ]:
import shutil

final = OUT / 'final'
assert (final / 'adapter_config.json').exists(), 'Adaptor final belum ada. Training belum selesai?'

evaluation = json.loads((OUT / 'evaluation.json').read_text(encoding='utf-8'))
manifest = json.loads((OUT / 'run_manifest.json').read_text(encoding='utf-8'))
print('GPU          :', manifest['gpu'])
print('Train/val    :', manifest['train_rows'], '/', manifest['validation_rows'])
print('Evaluasi     :', evaluation)

exports = Path('exports')
stage = exports / f'{RUN_NAME}_stage'
shutil.rmtree(stage, ignore_errors=True)
shutil.copytree(final, stage / 'qwen-kka-lora')
adapter_zip = shutil.make_archive(str(exports / f'{RUN_NAME}-qwen-kka-lora'), 'zip', stage)

records = exports / f'{RUN_NAME}_records'
shutil.rmtree(records, ignore_errors=True)
records.mkdir(parents=True)
for name in ['run_manifest.json', 'packages.lock.txt', 'evaluation.json', 'trainer_state.json']:
    if (OUT / name).exists():
        shutil.copy2(OUT / name, records / name)
records_zip = shutil.make_archive(str(exports / f'{RUN_NAME}-records'), 'zip', records)

for p in [adapter_zip, records_zip]:
    print(f'{p}  ({Path(p).stat().st_size / 2**20:.1f} MB)')

### Opsional: simpan ke Hugging Face Hub (private)

Cadangan selain unduhan manual. Butuh token **write** dari huggingface.co/settings/tokens.
Jangan menulis token langsung di notebook.

In [ ]:
# from huggingface_hub import login, HfApi
# login()  # tempel token saat diminta
# api = HfApi()
# repo_id = '<username>/qwen-kka-lora'
# api.create_repo(repo_id, private=True, exist_ok=True)
# api.upload_folder(folder_path=str(final), repo_id=repo_id, commit_message=RUN_NAME)

## 8. Pakai adaptor di laptop / server inferensi

1. Ekstrak `<run>-qwen-kka-lora.zip` ke folder `adapters/` proyek sehingga ada
   `adapters/qwen-kka-lora/adapter_config.json`.
2. Jalankan:
   ```bash
   python chat.py --mode terstruktur --adapter adapters/qwen-kka-lora
   python chat.py --mode umum --adapter adapters/qwen-kka-lora
   ```
   atau untuk API FastAPI, isi `MODEL_ADAPTER=adapters/qwen-kka-lora` di `.env`.

Adaptor tidak menggantikan model dasar; model dasar dan revisi pada `model_record.json`
harus sama. Inferensi di laptop tetap memakai fp16; adaptor yang dilatih dengan bf16 dapat
dimuat seperti biasa.

**Sebelum menghentikan instance:** pastikan kedua ZIP sudah terunduh.